# FinanceBench hybrid reranking demo

This notebook reads the saved genuine historical ten-case snapshot, displays a compact comparison, and renders the full evidence report. It is a saved demonstration snapshot, not a fresh run. FinanceBench reference answers are evaluation-only; Jev verdicts are model diagnostics, not accuracy labels.

In [ ]:
from pathlib import Path
import json, subprocess, sys
snapshot = Path('data/snapshot.json')
if not snapshot.exists():
    raise FileNotFoundError('Expected data/snapshot.json. Ask the project runner to export the saved 10-case snapshot.')
data = json.loads(snapshot.read_text(encoding='utf-8'))
rows = data.get('rows', [])
display_protocol = data.get('protocol', {})
print('Historical snapshot:', bool(display_protocol.get('historical')), '| model:', display_protocol.get('model', '—'))
len(rows), [r.get('id') for r in rows]

In [ ]:
from IPython.display import display, Markdown
table = [['ID', 'Company', 'Qwen3.5 answer', 'Reference answer', 'Jev verdict', 'Seconds']]
for r in rows:
    post = (r.get('postcheck') or {}).get('answers', {}).get('answer_verdict', {})
    table.append([str(r.get('id', '')), str(r.get('company', '')), str(r.get('answer', '')).replace('\n', ' '), str(r.get('reference_answer', '')).replace('\n', ' '), str(post.get('choice', '—')), str(r.get('generation_seconds', '—'))])
display(Markdown('\n'.join('| ' + ' | '.join(cell.replace('|', '\\|') for cell in row) + ' |' for row in [table[0], ['---'] * len(table[0])] + table[1:])))

In [ ]:
report = Path('data/latest-report.html')
subprocess.run([sys.executable, 'render.py', str(snapshot), '-o', str(report)], check=True)
display(Markdown(f'Report written to `{report}`. Open it in a browser to inspect retrieved passages.'))

## Re-run on a prepared machine

From PowerShell: `./setup.ps1`, then `python run.py --prepare`, then `python run.py --limit 10`. Set `AI_GATEWAY_API_KEY` in the environment when gateway access is needed; never paste it into this notebook.